In [1]:
import tensorflow as tf
print(f"TensorFlow версия: {tf.__version__}")
print(f"DirectML устройства: {tf.config.list_physical_devices()}")

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50, EfficientNetB0
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
import os
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
# Пути к данным
train_dir = 'D:/3cursETU/ML/lab1NN/Source/train'
val_dir = 'D:/3cursETU/ML/lab1NN/Source/validation'
test_dir = 'D:/3cursETU/ML/lab1NN/Source/test'

# Параметры
img_height, img_width = 224, 224
batch_size = 32

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
#Параметры
img_height, img_width = 224, 224
batch_size = 32
num_classes = 36
epochs = 30

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image

def fix_image(img):
    """Конвертирует изображение в RGB"""
    if img.mode != 'RGB':
        img = img.convert('RGB')
    return img

train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest',
    preprocessing_function=fix_image
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    preprocessing_function=fix_image
)


In [ ]:
#Визуализация аугментации
def visualize_augmentation(generator, num_samples=5):
    batch = next(generator)
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 6))
    
    for i in range(num_samples):
        # Оригинал
        axes[0, i].imshow(batch[0][i])
        axes[0, i].axis('off')
        axes[0, i].set_title('Original')
        
        # Аугментированное
        aug_batch = next(generator)
        axes[1, i].imshow(aug_batch[0][i])
        axes[1, i].axis('off')
        axes[1, i].set_title('Augmented')
    
    plt.tight_layout()
    plt.show()

visualize_augmentation(train_generator)

In [ ]:
#Архитектура 1 - Базовая CNN
def create_small_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', input_shape=(224,224,3)),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(64, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Conv2D(128, (3,3), activation='relu'),
        layers.MaxPooling2D(2,2),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(36, activation='softmax')
    ])
    return model

model_small = create_small_cnn()
model_small.summary()

In [ ]:
#Архитектура 2 - Улучшенная CNN
def create_medium_cnn():
    model = models.Sequential([
        layers.Conv2D(64, (3,3), activation='relu', padding='same', input_shape=(224,224,3)),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(2,2),
        
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(36, activation='softmax')
    ])
    return model

model_medium = create_medium_cnn()
model_medium.summary()

In [ ]:
# Ячейка 8: Архитектура 3 - Transfer Learning (ResNet50)
def create_transfer_model():
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=(224,224,3))
    base_model.trainable = False
    
    inputs = tf.keras.Input(shape=(224,224,3))
    x = tf.keras.applications.resnet50.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(36, activation='softmax')(x)
    
    model = tf.keras.Model(inputs, outputs)
    return model

model_transfer = create_transfer_model()
model_transfer.summary()

In [ ]:
#Функция для обучения модели
def train_model(model, model_name, train_gen, val_gen, epochs=30):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3),
        ModelCheckpoint(f'best_{model_name}.h5', monitor='val_accuracy', save_best_only=True)
    ]
    
    print(f"\nОбучение модели {model_name}...")
    history = model.fit(
        train_gen,
        validation_data=val_gen,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    
    return history

# Обучите каждую модель (это займет время)
# history_small = train_model(model_small, "small_cnn", train_generator, val_generator)
# history_medium = train_model(model_medium, "medium_cnn", train_generator, val_generator)
# history_transfer = train_model(model_transfer, "transfer_resnet50", train_generator, val_generator)

In [ ]:
history_small = train_model(model_small, "small_cnn", train_generator, val_generator)